# GrC ASC-only NEURON / BrainCell comparison


In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("JAX_PLATFORMS", "cpu")

def find_repo_root(start=None):
    cwd = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "braincell").exists() and (candidate / "examples").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or a subdirectory inside it.")

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import brainunit as u
import brainstate
from braincell import mech
from braincell.filter import at
from neuron import h

from examples.neuron_compare.cell.grc_ma2020.parameters import DEFAULT_MORPH_PATH, DEFAULT_NRNMECH_PATH, load_grc20_params
from examples.neuron_compare.cell.grc_ma2020.grc_neuron import GrC as NeuronGrC
from examples.neuron_compare.cell.grc_ma2020.grc_braincell import GrC as BrainCellGrC

brainstate.environ.set(precision=64)


In [ ]:
params = load_grc20_params()
dt_ms = 0.1
duration_ms = 50.0
delay_ms = 10.0
stim_dur_ms = 30.0
amp_nA = 0.01
temperature_celsius = 25.0
v_init_mV = -65.0


In [ ]:
neuron_grc = NeuronGrC(
    DEFAULT_MORPH_PATH,
    params=params,
    nrnmech_path=DEFAULT_NRNMECH_PATH,
    temperature_celsius=temperature_celsius,
).build()
stim = h.IClamp(neuron_grc.root_soma(0.5))
stim.delay = delay_ms
stim.dur = stim_dur_ms
stim.amp = amp_nA
t_neuron = h.Vector().record(h._ref_t)
v_soma_neuron = h.Vector().record(neuron_grc.root_soma(0.5)._ref_v)
h.cvode_active(0)
h.dt = dt_ms
h.steps_per_ms = 1.0 / h.dt
h.celsius = temperature_celsius
h.tstop = duration_ms
h.v_init = v_init_mV
h.finitialize(h.v_init)
h.run()

braincell_grc = BrainCellGrC(DEFAULT_MORPH_PATH, params=params, temperature_celsius=temperature_celsius, v_init_mV=v_init_mV).build()
braincell_grc.cell.place(at("soma", 0.5), mech.StateProbe(name="v_soma"))
braincell_grc.cell.place(at("soma", 0.5), mech.CurrentClamp.step(amp_nA * u.nA, stim_dur_ms * u.ms, delay=delay_ms * u.ms))
braincell_grc.cell.init_state()
braincell_grc.cell.reset_state()
braincell_run = braincell_grc.cell.run(dt=dt_ms * u.ms, duration=duration_ms * u.ms)


In [ ]:
neuron_t = np.asarray(t_neuron, dtype=float).reshape(-1)
neuron_v = np.asarray(v_soma_neuron, dtype=float).reshape(-1)
braincell_v = np.asarray(braincell_run.traces["v_soma"].to_decimal(u.mV), dtype=float).reshape(-1)
braincell_t = (np.arange(braincell_v.shape[0], dtype=float) + 1.0) * dt_ms

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(neuron_t, neuron_v, label="NEURON soma")
ax.plot(braincell_t[: braincell_v.shape[0]], braincell_v, label="BrainCell soma", linestyle="--")
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Voltage (mV)")
ax.legend()
ax.set_title("GrC ASC-only soma voltage")


In [ ]:
# Optional cleanup when re-running cells in the same kernel.
# neuron_grc.cleanup()
